In [7]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
from scipy.stats import spearmanr, pearsonr
import ts_onset_cess as ocd
import pandas as pd
from fapar_def import fapar_read

import warnings
warnings.filterwarnings('ignore')

In [8]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
dataf = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/data/analysed_tropomi/"

### Read in precip 
- do this for whole available ts and then sub-select years in next step for analysis

In [9]:
Y1=2018
Y2=2024

dD_all_list = []
for Y in range(Y1,Y2+1):
    dD = xr.open_mfdataset('/Volumes/blue_wd/ESA_F4R/tropomi_merged/TROPOMI_merged_'+str(Y)+'.nc')
    dD = dD.where((dD.lat < 5.) & (dD.lat > -5.) & (dD.lon < 29.) & (dD.lon > 8.),drop=True)
    dD = dD['deltad']
    dD_year_list = []
    for m in range(1,13):
        try:
            mp = dD.sel(time=(dD.time.dt.month==m), drop=True)
            #print(mp)
            bins = [mp.time[0].values,mp.time[10-1].values,mp.time[20-1].values,mp.time[-1].values]
            mp_out = mp.groupby_bins('time', bins,labels=[mp.time[10-1].values,mp.time[20-1].values,mp.time[-1].values]).mean()
            mp_out = mp_out.rename({'time_bins':'time'})
            #print(mp_out)
            dD_year_list.append(mp_out)
        except:
            print('no month - ',m,' for year - ',Y)
    dD_year = xr.concat(dD_year_list,dim='time')
    dD_all_list.append(dD_year)
    dD.close()
    print('done - ',Y)
dD_all = xr.concat(dD_all_list,dim='time')
dD_all = dD_all.sel(time=slice('2018-07-01','2024-12-31'))
print(dD_all)
    
dD_all.to_netcdf(dataf+'tropomi_10day_eq.nc',engine='h5netcdf')
        

KeyboardInterrupt: 